# Analytics Layer - Player Performance Metrics

This notebook creates **analytics tables** on top of the gold layer, providing aggregated views and performance metrics for fantasy football analysis.

## Objectives

1. **Weekly Aggregations** - Average points, consistency scores, participation rates
2. **Season Statistics** - Cumulative stats, season-to-date rankings
3. **Player Trends** - Hot/cold streaks, momentum indicators, recent form
4. **Positional Rankings** - Percentile rankings within positions
5. **Multi-Source Comparison** - Data quality and coverage analysis

## Tables Created

| Table | Type | Purpose |
|-------|------|----------|
| `analytics_player_weekly_agg` | Aggregation | Weekly performance summary per player |
| `analytics_player_season_stats` | Aggregation | Season-to-date cumulative statistics |
| `analytics_player_trends` | Metrics | Momentum, streaks, recent form (last 3/5 games) |
| `analytics_positional_rankings` | Rankings | Percentile rankings within each position |
| `analytics_source_coverage` | Quality | Multi-source data coverage and completeness |

## Data Flow

**Gold Layer** → **Analytics Layer** → **ML Features / Dashboards**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import json
from datetime import datetime

print("✓ Imports loaded")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"

print(f"\n✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")

# Load gold tables
player_dim = spark.table(f"{CATALOG}.{SCHEMA}.gold_player_dim")
gold_stats = spark.table(f"{CATALOG}.{SCHEMA}.gold_weekly_stats")

print(f"\n✓ Loaded gold_player_dim: {player_dim.count()} players")
print(f"✓ Loaded gold_weekly_stats: {gold_stats.count()} records")

In [0]:
# Create weekly aggregations for each player
print("="*70)
print("CREATING WEEKLY PLAYER AGGREGATIONS")
print("="*70)

# Calculate weekly metrics
weekly_agg = gold_stats.groupBy(
    "master_player_id",
    "season",
    "week"
).agg(
    # Basic stats
    F.first("player_name").alias("player_name"),
    F.first("position").alias("position"),
    F.first("team").alias("team"),
    
    # Fantasy points from all sources
    F.avg("fantasy_points").alias("avg_fantasy_points"),
    F.max("fantasy_points").alias("max_fantasy_points"),
    F.min("fantasy_points").alias("min_fantasy_points"),
    F.stddev("fantasy_points").alias("stddev_fantasy_points"),
    
    # Source coverage
    F.count("source").alias("source_count"),
    F.collect_set("source").alias("sources"),
    
    # Data quality
    F.max("ingested_at").alias("last_updated")
).withColumn(
    # Consistency score: lower stddev = more consistent across sources
    "consistency_score",
    F.when(
        F.col("stddev_fantasy_points").isNull(),
        1.0
    ).otherwise(
        F.when(
            F.col("stddev_fantasy_points") == 0,
            1.0
        ).otherwise(
            F.lit(1.0) / (F.lit(1.0) + F.col("stddev_fantasy_points"))
        )
    )
)

print(f"\n✓ Calculated weekly aggregations: {weekly_agg.count()} player-weeks")

# Write to table
weekly_agg.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.analytics_player_weekly_agg"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.analytics_player_weekly_agg")

# Show sample
print("\nSample weekly aggregations:")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_weekly_agg")
    .orderBy(F.desc("avg_fantasy_points"))
    .limit(20)
)

In [0]:
# Create season-to-date cumulative statistics
print("="*70)
print("CREATING SEASON-TO-DATE STATISTICS")
print("="*70)

# Load weekly aggregations
weekly_agg = spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_weekly_agg")

# Calculate season stats
season_stats = weekly_agg.groupBy(
    "master_player_id",
    "season"
).agg(
    # Identity
    F.first("player_name").alias("player_name"),
    F.first("position").alias("position"),
    F.last("team").alias("current_team"),
    
    # Performance metrics
    F.sum("avg_fantasy_points").alias("total_fantasy_points"),
    F.avg("avg_fantasy_points").alias("avg_points_per_game"),
    F.max("avg_fantasy_points").alias("best_game"),
    F.min("avg_fantasy_points").alias("worst_game"),
    F.stddev("avg_fantasy_points").alias("stddev_points"),
    
    # Participation
    F.count("week").alias("games_played"),
    F.min("week").alias("first_week"),
    F.max("week").alias("last_week"),
    
    # Data quality
    F.avg("source_count").alias("avg_sources_per_week"),
    F.max("last_updated").alias("last_updated")
).withColumn(
    # Performance consistency (coefficient of variation)
    "performance_consistency",
    F.when(
        F.col("avg_points_per_game") > 0,
        F.col("stddev_points") / F.col("avg_points_per_game")
    ).otherwise(None)
).withColumn(
    # Boom/bust potential (range / avg)
    "boom_bust_ratio",
    F.when(
        F.col("avg_points_per_game") > 0,
        (F.col("best_game") - F.col("worst_game")) / F.col("avg_points_per_game")
    ).otherwise(None)
)

print(f"\n✓ Calculated season stats: {season_stats.count()} player-seasons")

# Write to table
season_stats.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.analytics_player_season_stats"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.analytics_player_season_stats")

# Show top performers
print("\nTop 20 players by total fantasy points:")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_season_stats")
    .filter(F.col("season") == 2025)
    .orderBy(F.desc("total_fantasy_points"))
    .limit(20)
)

In [0]:
# Calculate player trends: hot/cold streaks, momentum, recent form
print("="*70)
print("CALCULATING PLAYER TRENDS AND MOMENTUM")
print("="*70)

# Load weekly aggregations
weekly_agg = spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_weekly_agg")

# Create window for each player-season ordered by week
window_spec = Window.partitionBy("master_player_id", "season").orderBy("week")

# Calculate rolling averages and trends
trends = weekly_agg.withColumn(
    # Last 3 games average
    "last_3_games_avg",
    F.avg("avg_fantasy_points").over(
        window_spec.rowsBetween(-2, 0)
    )
).withColumn(
    # Last 5 games average
    "last_5_games_avg",
    F.avg("avg_fantasy_points").over(
        window_spec.rowsBetween(-4, 0)
    )
).withColumn(
    # Season average up to this point
    "season_avg_to_date",
    F.avg("avg_fantasy_points").over(
        window_spec.rowsBetween(Window.unboundedPreceding, 0)
    )
).withColumn(
    # Momentum: difference between last 3 and season average
    "momentum_score",
    F.col("last_3_games_avg") - F.col("season_avg_to_date")
).withColumn(
    # Trend direction: is player improving or declining?
    "trend_direction",
    F.when(
        F.col("momentum_score") > 2.0, "Hot"
    ).when(
        F.col("momentum_score") < -2.0, "Cold"
    ).otherwise("Stable")
).withColumn(
    # Week-over-week change
    "wow_change",
    F.col("avg_fantasy_points") - F.lag("avg_fantasy_points", 1).over(window_spec)
).withColumn(
    # Consecutive weeks with points > 10 (starting threshold)
    "scoring_streak",
    F.when(
        F.col("avg_fantasy_points") >= 10.0,
        F.count(F.when(F.col("avg_fantasy_points") >= 10.0, 1)).over(
            window_spec.rowsBetween(Window.unboundedPreceding, 0)
        )
    ).otherwise(0)
).select(
    "master_player_id",
    "season",
    "week",
    "player_name",
    "position",
    "team",
    "avg_fantasy_points",
    "last_3_games_avg",
    "last_5_games_avg",
    "season_avg_to_date",
    "momentum_score",
    "trend_direction",
    "wow_change",
    "scoring_streak"
)

print(f"\n✓ Calculated trends: {trends.count()} player-weeks")

# Write to table
trends.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.analytics_player_trends"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.analytics_player_trends")

# Show hottest players (most recent week)
print("\nHottest players (positive momentum, most recent week):")
max_week = trends.agg(F.max("week")).collect()[0][0]
display(
    spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_trends")
    .filter((F.col("season") == 2025) & (F.col("week") == max_week))
    .filter(F.col("trend_direction") == "Hot")
    .orderBy(F.desc("momentum_score"))
    .limit(20)
)

In [0]:
# Create positional rankings and percentile scores
print("="*70)
print("CREATING POSITIONAL RANKINGS")
print("="*70)

# Load season stats
season_stats = spark.table(f"{CATALOG}.{SCHEMA}.analytics_player_season_stats")

# Create window for ranking within position
position_window = Window.partitionBy("season", "position").orderBy(F.desc("total_fantasy_points"))

# Calculate rankings
rankings = season_stats.withColumn(
    # Overall rank within position
    "position_rank",
    F.row_number().over(position_window)
).withColumn(
    # Total players in position
    "total_in_position",
    F.count("*").over(Window.partitionBy("season", "position"))
).withColumn(
    # Percentile (0-100, higher is better)
    "percentile",
    F.round(
        (F.lit(1.0) - (F.col("position_rank") - F.lit(1.0)) / F.col("total_in_position")) * F.lit(100),
        1
    )
).withColumn(
    # Tier classification
    "tier",
    F.when(F.col("percentile") >= 90, "Elite")
    .when(F.col("percentile") >= 70, "Strong")
    .when(F.col("percentile") >= 50, "Average")
    .when(F.col("percentile") >= 30, "Below Average")
    .otherwise("Waiver Wire")
).select(
    "master_player_id",
    "season",
    "player_name",
    "position",
    "current_team",
    "total_fantasy_points",
    "avg_points_per_game",
    "games_played",
    "position_rank",
    "total_in_position",
    "percentile",
    "tier",
    "performance_consistency",
    "boom_bust_ratio"
)

print(f"\n✓ Calculated rankings: {rankings.count()} player-seasons")

# Write to table
rankings.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.analytics_positional_rankings"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.analytics_positional_rankings")

# Show elite players by position
print("\nElite players by position (2025 season):")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.analytics_positional_rankings")
    .filter((F.col("season") == 2025) & (F.col("tier") == "Elite"))
    .orderBy("position", "position_rank")
    .limit(50)
)

In [0]:
# Analyze data coverage and quality across sources
print("="*70)
print("MULTI-SOURCE DATA COVERAGE ANALYSIS")
print("="*70)

# Load gold stats and player dim
gold_stats = spark.table(f"{CATALOG}.{SCHEMA}.gold_weekly_stats")
player_dim = spark.table(f"{CATALOG}.{SCHEMA}.gold_player_dim")

# Calculate source coverage per player
source_coverage = gold_stats.groupBy(
    "master_player_id",
    "season"
).agg(
    F.first("player_name").alias("player_name"),
    F.first("position").alias("position"),
    
    # Overall stats
    F.countDistinct("source").alias("unique_sources"),
    F.collect_set("source").alias("sources"),
    F.count("*").alias("total_records"),
    F.countDistinct("week").alias("weeks_covered"),
    
    # Per-source record counts
    F.sum(F.when(F.col("source") == "espn_public", 1).otherwise(0)).alias("espn_records"),
    F.sum(F.when(F.col("source") == "nflverse", 1).otherwise(0)).alias("nflverse_records"),
    F.sum(F.when(F.col("source") == "sleeper", 1).otherwise(0)).alias("sleeper_records"),
    F.sum(F.when(F.col("source") == "api_sports", 1).otherwise(0)).alias("api_sports_records"),
    F.sum(F.when(F.col("source") == "fantasai", 1).otherwise(0)).alias("fantasai_records"),
    
    # Data quality
    F.max("ingested_at").alias("last_updated")
).withColumn(
    # Coverage score: how many sources have data for this player
    "coverage_score",
    F.col("unique_sources") / F.lit(5.0)  # Out of 5 possible sources
).withColumn(
    # Data completeness: records per week
    "records_per_week",
    F.col("total_records") / F.col("weeks_covered")
)

print(f"\n✓ Calculated source coverage: {source_coverage.count()} player-seasons")

# Write to table
source_coverage.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.analytics_source_coverage"
)

print(f"\n✓ Created table: {CATALOG}.{SCHEMA}.analytics_source_coverage")

# Show players with best data coverage
print("\nPlayers with best data coverage (multiple sources):")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.analytics_source_coverage")
    .filter(F.col("season") == 2025)
    .orderBy(F.desc("unique_sources"), F.desc("weeks_covered"))
    .limit(20)
)

# Summary statistics
print("\nSource coverage summary by position:")
coverage_summary = spark.sql(f"""
    SELECT 
        position,
        COUNT(DISTINCT master_player_id) as total_players,
        AVG(unique_sources) as avg_sources,
        AVG(coverage_score) as avg_coverage_score,
        SUM(espn_records) as total_espn,
        SUM(nflverse_records) as total_nflverse,
        SUM(sleeper_records) as total_sleeper,
        SUM(api_sports_records) as total_api_sports,
        SUM(fantasai_records) as total_fantasai
    FROM {CATALOG}.{SCHEMA}.analytics_source_coverage
    WHERE season = 2025
    GROUP BY position
    ORDER BY total_players DESC
""")

display(coverage_summary)

In [0]:
%sql
-- Sample analytics queries for quick insights

-- 1. Top performers by position (2025)
SELECT 
    position,
    player_name,
    current_team,
    total_fantasy_points,
    avg_points_per_game,
    games_played,
    tier
FROM main.fantasai.analytics_positional_rankings
WHERE season = 2025 
    AND position_rank <= 10
    AND position IN ('QB', 'RB', 'WR', 'TE')
ORDER BY position, position_rank
LIMIT 40;

-- 2. Most consistent players (low variance, high avg)
SELECT 
    r.player_name,
    r.position,
    r.current_team,
    r.avg_points_per_game,
    r.performance_consistency,
    r.games_played,
    r.tier
FROM main.fantasai.analytics_positional_rankings r
WHERE r.season = 2025
    AND r.games_played >= 5
    AND r.avg_points_per_game >= 8.0
    AND r.performance_consistency IS NOT NULL
ORDER BY r.performance_consistency ASC
LIMIT 30;

-- 3. Trending up players (recent hot streaks)
SELECT 
    t.player_name,
    t.position,
    t.team,
    t.avg_fantasy_points as this_week_points,
    t.last_3_games_avg,
    t.season_avg_to_date,
    t.momentum_score,
    t.trend_direction
FROM main.fantasai.analytics_player_trends t
WHERE t.season = 2025
    AND t.week = (SELECT MAX(week) FROM main.fantasai.analytics_player_trends WHERE season = 2025)
    AND t.trend_direction = 'Hot'
ORDER BY t.momentum_score DESC
LIMIT 30;